---
# Perbandingan Pendekatan CBR Berbasis TF-IDF: Cosine Similarity vs. SVM dalam Analisis Putusan Pengadilan

Anggota Kelompok :

1. Mokh. Brillian Dwi Ariestianto - 202210370311442

2. Muhammad Razzan Carveyna Hibrizi - 202210370311445

## Tugas Besar Mata Kuliah Penalaran Komputer (A)
---



# Tahap 1 – Membangun Case Base

## Tujuan

Tahap ini bertujuan untuk membangun basis kasus (case base) awal dengan cara mengumpulkan, mengekstrak, dan membersihkan dokumen putusan pengadilan dari sumber resmi Mahkamah Agung Republik Indonesia. Hasil akhir dari tahap ini berupa teks putusan yang telah dibersihkan dan siap diproses lebih lanjut dalam siklus Case-Based Reasoning (CBR).

---

## Langkah Kerja

### 1. Seleksi dan Pengunduhan Dokumen

- Domain perkara yang dipilih: **Pidana Umum – Pembunuhan (PN MEDAN)**
- Sumber dokumen: Direktori Putusan Mahkamah Agung Republik Indonesia
- Format dokumen: PDF
- Jumlah dokumen: **47 dokumen**

Dokumen diunduh secara manual dan disimpan dalam folder `pdf_downloaded/`.

---

### 2. Konversi dan Ekstraksi Teks

Setiap file PDF dikonversi menjadi teks polos (plain text) menggunakan pustaka `pdfminer`. Tujuan dari konversi ini adalah untuk memperoleh isi putusan dalam format yang bisa diproses lebih lanjut.

---

### 3. Pembersihan Teks (Cleaning)

Teks hasil ekstraksi dibersihkan dengan cara:

- Menghapus watermark, header, footer, dan nomor halaman
- Menghapus konten disclaimer dari MA RI
- Menormalkan spasi dan huruf (lowercase)
- Menghitung rasio keutuhan dokumen (panjang teks bersih dibanding teks awal)

Dokumen hanya disimpan jika memenuhi syarat minimal rasio keutuhan ≥ 80%.

---

### 4. Validasi dan Logging

Semua dokumen dicatat dalam log proses pembersihan (`logs/cleaning.log`) dengan informasi rasio keutuhan per kasus. Log ini berguna untuk memantau kelayakan data dan mendeteksi dokumen bermasalah.

---

## Output Tahap Ini

- Folder `/data/raw/*.txt` berisi 47 file teks putusan yang telah dibersihkan dan lolos validasi.
- File log `/logs/cleaning.log` berisi rekaman validasi keutuhan untuk setiap kasus.
- Semua dokumen memiliki rasio keutuhan di atas 88%, menandakan proses ekstraksi dan cleaning berhasil dilakukan dengan baik.

Contoh log validasi:
[OK] case_001 diproses (89.08% valid)
[OK] case_002 diproses (89.29% valid)
[OK] case_003 diproses (89.45% valid)
...
[OK] case_047 diproses (89.34% valid)


---
Tahap pertama ini berhasil menyiapkan kumpulan kasus dengan kualitas teks yang layak untuk digunakan sebagai basis kasus pada sistem CBR. Tahapan ini menjadi fondasi penting untuk proses representasi dan retrieval pada tahap-tahap selanjutnya.



In [44]:
import os
import re
import glob
import string  # ← tambahkan ini
from pdfminer.high_level import extract_text
from datetime import datetime


# === Konfigurasi path ===
PDF_FOLDER = 'pdf_downloaded'
OUTPUT_FOLDER = 'data/raw'
LOG_FILE = 'logs/cleaning.log'

def clean_text(text):

    baseline_length = len(text)
   
    # Hapus header/footer watermark
    text = text.replace("Direktori Putusan Mahkamah Agung Republik Indonesia", "")
    text = text.replace("putusan.mahkamahagung.go.id", "")
    text = re.sub(r'halaman\s*\d+', '', text, flags=re.IGNORECASE)
    text = text.replace("M a h ka m a h A g u n g R e p u blik In d o n esia\n", "")
    cleaned_length = len(text)
    text = text.replace("Disclaimer\n", "")
    text = text.replace(
        "Kepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas\n", "")
    text = text.replace(
        "pelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.\n", "")
    text = text.replace(
        "Dalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi Kepaniteraan Mahkamah Agung RI melalui :\n", "")
    text = text.replace(
        "Email : kepaniteraan@mahkamahagung.go.id    Telp : 021-384 3348 (ext.318)\n", "")
    


    # Normalisasi akhir
    text = text.lower()
    text = ' '.join(text.split())

    ratio = cleaned_length / baseline_length 

    return text, ratio



# === Fungsi log (opsional) ===
def write_log(case_name, ratio):
    os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        timestamp = datetime.now().isoformat()
        f.write(f"[{timestamp}] {case_name} | Integrity: {ratio:.2%}\n")

# === Proses utama ===
def process_pdfs():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    pdf_files = sorted(glob.glob(os.path.join(PDF_FOLDER, '*.pdf')))
    
    for i, pdf_path in enumerate(pdf_files):
        case_id = f"case_{i+1:03d}"
        output_file = os.path.join(OUTPUT_FOLDER, f"{case_id}.txt")

        try:
            # Ekstrak teks
            text = extract_text(pdf_path)

            # Bersihkan
            cleaned_text, ratio = clean_text(text)

            # Validasi keutuhan
            if ratio < 0.8:
                print(f"[WARNING] {case_id} hanya {ratio:.2%} isi yang tersisa.")
            else:
                print(f"[OK] {case_id} diproses ({ratio:.2%} valid).")

                # Simpan teks bersih HANYA jika valid
                with open(output_file, 'w', encoding='utf-8') as f_out:
                    f_out.write(cleaned_text)

            # Catat log tetap dicatat semuanya
            write_log(case_id, ratio)

        except Exception as e:
            print(f"[ERROR] Gagal memproses {pdf_path}: {e}")

# === Eksekusi utama ===
if __name__ == "__main__":
    process_pdfs()

[OK] case_001 diproses (89.08% valid).
[OK] case_002 diproses (89.29% valid).
[OK] case_003 diproses (89.45% valid).
[OK] case_004 diproses (88.65% valid).
[OK] case_005 diproses (88.92% valid).
[OK] case_006 diproses (87.81% valid).
[OK] case_007 diproses (88.64% valid).
[OK] case_008 diproses (88.58% valid).
[OK] case_009 diproses (88.47% valid).
[OK] case_010 diproses (88.78% valid).
[OK] case_011 diproses (89.01% valid).
[OK] case_012 diproses (90.03% valid).
[OK] case_013 diproses (88.01% valid).
[OK] case_014 diproses (88.94% valid).
[OK] case_015 diproses (88.12% valid).
[OK] case_016 diproses (89.84% valid).
[OK] case_017 diproses (88.83% valid).
[OK] case_018 diproses (89.26% valid).
[OK] case_019 diproses (89.14% valid).
[OK] case_020 diproses (88.84% valid).
[OK] case_021 diproses (89.68% valid).
[OK] case_022 diproses (89.60% valid).
[OK] case_023 diproses (89.56% valid).
[OK] case_024 diproses (89.76% valid).
[OK] case_025 diproses (89.76% valid).
[OK] case_026 diproses (8

# Tahap 2 – Case Representation

## Tujuan

Tahapan ini bertujuan untuk merepresentasikan setiap putusan dalam struktur data yang terorganisir. Hasil representasi ini menjadi basis data terstruktur yang siap digunakan untuk proses retrieval dan analisis lebih lanjut dalam sistem Case-Based Reasoning (CBR).

---

## Langkah Kerja

### 1. Ekstraksi Metadata

Setiap dokumen hasil cleaning dianalisis untuk mengekstrak informasi penting sebagai metadata, meliputi:

- Nomor Perkara (`no_perkara`)
- Tanggal Putusan (`tanggal`)
- Ringkasan Fakta (`ringkasan_fakta`)
- Pasal yang didakwakan (`pasal`)
- Pihak terkait (Terdakwa dan Korban)
- Isi teks lengkap (`text_full`)

Ekstraksi dilakukan dengan pendekatan berbasis pola (regex) terhadap isi dokumen.

---

### 2. Penyimpanan Data Terstruktur

Data hasil ekstraksi disimpan dalam dua format:

- **CSV**: `data/processed/cases_extracted.csv`
- **JSON**: `data/processed/cases_extracted.json`

Struktur kolom yang digunakan meliputi:

- `case_id`
- `no_perkara`
- `tanggal`
- `ringkasan_fakta`
- `pasal`
- `pihak`
- `text_full`

Jumlah data yang berhasil diproses: **47 kasus**

Contoh output terminal:

[SUKSES] 47 kasus disimpan ke:

CSV → data/processed/cases_extracted.csv

JSON → data/processed/cases_extracted.json


---

### 3. Feature Engineering

Untuk meningkatkan pemanfaatan data kasus, dilakukan proses rekayasa fitur (feature engineering) yang meliputi:

- **Jumlah Kata (Length)**: Menghitung total token (kata) dalam teks.
- **Bag-of-Words (BoW)**: Menghitung frekuensi kata dalam setiap kasus.
- **QA-Pairs Sederhana**: Menghasilkan pasangan pertanyaan dan jawaban dari konten teks.

QA-Pairs mencakup contoh pertanyaan berikut:

- Apa nomor perkaranya?
- Apa pasal yang dilanggar?
- Siapa terdakwanya?
- Siapa korbannya?

---

### 4. Penyimpanan Fitur

Hasil rekayasa fitur disimpan dalam format JSON:

- `data/processed/features_length.json`
- `data/processed/features_bow.json`
- `data/processed/features_qa_pairs.json`

Contoh output terminal:

[SUKSES] Feature Engineering selesai!

Length disimpan di : data/processed/features_length.json

Bag-of-Words disimpan di: data/processed/features_bow.json

QA-pairs disimpan di : data/processed/features_qa_pairs.json


---

Tahap representasi berhasil membentuk struktur data terorganisir untuk 47 kasus. Setiap kasus dilengkapi metadata, ringkasan fakta, dan fitur tambahan untuk mendukung proses retrieval dan prediksi pada tahapan selanjutnya dalam sistem CBR.


In [8]:
import os
import re
import json
import pandas as pd

# === Path Konfigurasi ===
RAW_FOLDER = 'data/raw'
CSV_OUTPUT = 'data/processed/cases_extracted.csv'
JSON_OUTPUT = 'data/processed/cases_extracted.json'
os.makedirs('data/processed', exist_ok=True)

# === Fungsi pencarian hasil pertama ===
def find_first(pattern, text):
    results = re.findall(pattern, text, re.IGNORECASE | re.DOTALL)
    return results[0].strip() if results else ""

# === Fungsi membersihkan nilai CSV ===
def clean_for_csv(text):
    if not text:
        return ""
    return text.replace(",", " ").replace("\n", " ").strip()

# === Ekstraksi data utama per dokumen ===
def extract_case_data(case_id, text):
    text_clean = " ".join(text.split())

    no_perkara = clean_for_csv(find_first(r"p\s*u\s*t\s*u\s*s\s*a\s*n\s*\"?(.*?)\"?\s*demi keadilan", text_clean))
    tanggal = find_first(r'pengadilan negeri medan, pada hari\s+"?([^"]+)"?\s*,?\s*oleh', text_clean)
    ringkasan_fakta = find_first(r'fakta-fakta\s+"(.*?)"\s*,?\s*menimbang, bahwa', text_clean)
    pasal = find_first(r'melanggar pasal\s*"?([^"]+?)"?[\.\,\:\;]', text_clean)

    # Terdakwa
    pihak_terdakwa = find_first(r'menjatuhkan pidana terhadap\s+"([^"]+)"\s+(?:dengan pidana|3\. menyatakan)', text_clean)

    # Korban: ambil satu kata setelah 'korban '
    korban_match = find_first(r'korban\s+([a-zA-Z]+)', text_clean)

    # Gabungkan pihak
    pihak = f"Terdakwa: {pihak_terdakwa} Korban: {korban_match}" if pihak_terdakwa or korban_match else ""

    return {
        "case_id": case_id,
        "no_perkara": clean_for_csv(no_perkara),
        "tanggal": clean_for_csv(tanggal),
        "ringkasan_fakta": clean_for_csv(ringkasan_fakta),
        "pasal": clean_for_csv(pasal),
        "pihak": clean_for_csv(pihak),
        "text_full": clean_for_csv(text)
    }

# === Proses semua dokumen ===
cases = []
for i in range(1, 48):  # case_001.txt - case_047.txt
    filepath = os.path.join(RAW_FOLDER, f"case_{i:03}.txt")
    if not os.path.exists(filepath):
        continue

    with open(filepath, encoding='utf-8') as file:
        text = file.read()
        case_data = extract_case_data(i, text)
        cases.append(case_data)

# === Simpan ke CSV
df = pd.DataFrame(cases)
df.to_csv(CSV_OUTPUT, index=False)

# === Simpan ke JSON
with open(JSON_OUTPUT, 'w', encoding='utf-8') as jf:
    json.dump(cases, jf, ensure_ascii=False, indent=2)

print(f"[SUKSES] {len(cases)} kasus disimpan ke:")
print(f"- CSV  → {CSV_OUTPUT}")
print(f"- JSON → {JSON_OUTPUT}")


[SUKSES] 47 kasus disimpan ke:
- CSV  → data/processed/cases_extracted.csv
- JSON → data/processed/cases_extracted.json


In [9]:
import os
import json
import re
from collections import Counter

# === Path ===
RAW_FOLDER = 'data/raw'
PROCESSED_FOLDER = 'data/processed'
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

# === File Output Feature Engineering ===
LENGTH_FILE = os.path.join(PROCESSED_FOLDER, 'features_length.json')
BOW_FILE = os.path.join(PROCESSED_FOLDER, 'features_bow.json')
QA_FILE = os.path.join(PROCESSED_FOLDER, 'features_qa_pairs.json')

# === Tokenizer sederhana ===
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

# === Ekstraksi QA pairs ===
def get_qa_pairs(text):
    return {
        "Apa nomor perkaranya?": re.search(r"putusan\s+\"?([^\"\,\n]+)", text, re.IGNORECASE | re.DOTALL),
        "Apa pasal yang dilanggar?": re.search(r"melanggar pasal\s+\"?([^\"\,\.\;\:]+)", text, re.IGNORECASE),
        "Siapa terdakwanya?": re.search(r"terdakwa\s+([a-zA-Z]+)", text, re.IGNORECASE),
        "Siapa korbannya?": re.search(r"korban\s+([a-zA-Z]+)", text, re.IGNORECASE),
    }

# === Proses semua file ===
length_data = {}
bow_data = {}
qa_data = {}

for i in range(1, 48):  # case_001.txt - case_047.txt
    file_path = os.path.join(RAW_FOLDER, f"case_{i:03}.txt")
    if not os.path.exists(file_path):
        continue

    with open(file_path, encoding='utf-8') as f:
        raw_text = f.read()

    text_clean = " ".join(raw_text.split())
    tokens = tokenize(text_clean)
    case_id = f"case_{i:03}"

    # Jumlah kata
    length_data[case_id] = len(tokens)

    # Bag-of-Words
    bow_data[case_id] = dict(Counter(tokens))

    # QA Pairs
    qas = get_qa_pairs(text_clean)
    qa_data[case_id] = {q: m.group(1).strip() if m else None for q, m in qas.items()}

# === Simpan ke file JSON terpisah ===
with open(LENGTH_FILE, 'w', encoding='utf-8') as f:
    json.dump(length_data, f, ensure_ascii=False, indent=2)

with open(BOW_FILE, 'w', encoding='utf-8') as f:
    json.dump(bow_data, f, ensure_ascii=False, indent=2)

with open(QA_FILE, 'w', encoding='utf-8') as f:
    json.dump(qa_data, f, ensure_ascii=False, indent=2)

print("[SUKSES] Feature Engineering selesai!")
print(f"- Length disimpan di      : {LENGTH_FILE}")
print(f"- Bag-of-Words disimpan di: {BOW_FILE}")
print(f"- QA-pairs disimpan di    : {QA_FILE}")


[SUKSES] Feature Engineering selesai!
- Length disimpan di      : data/processed\features_length.json
- Bag-of-Words disimpan di: data/processed\features_bow.json
- QA-pairs disimpan di    : data/processed\features_qa_pairs.json


# Tahap 3 – Case Retrieval

## Tujuan

Tahap ini bertujuan untuk menemukan kasus-kasus lama yang paling relevan dan mirip dengan query kasus baru yang diajukan. Proses ini merupakan bagian utama dalam sistem Case-Based Reasoning (CBR) untuk mendukung analisis dan pencarian preseden hukum.

---

## Langkah Kerja

### 1. Representasi Vektor

- Setiap ringkasan fakta dari putusan diubah menjadi representasi vektor menggunakan algoritma **TF-IDF** (`TfidfVectorizer` dari `sklearn`).
- Alternatif lain yang tersedia namun tidak digunakan pada tahap ini adalah embedding berbasis **transformer** seperti **IndoBERT**.

### 2. Splitting Data

- Dataset dibagi menjadi dua bagian: **data latih (train)** dan **data uji (test)** dengan rasio **80:20**.
- Teknik ini digunakan untuk pelatihan model klasifikasi berbasis TF-IDF + SVM.

### 3. Model Retrieval

Dalam tahap ini, sistem dibangun menggunakan **dua pendekatan berbeda** untuk melakukan retrieval terhadap kasus lama yang paling relevan dengan query baru:

#### a. TF-IDF + Cosine Similarity (Pendekatan Case-Based Reasoning)

- Menggunakan **TF-IDF vectorizer** untuk merepresentasikan teks ringkasan fakta dari semua kasus sebagai vektor numerik.
- Query kasus baru juga diubah menjadi vektor menggunakan TF-IDF yang sama.
- Kemiripan antar vektor dihitung menggunakan **cosine similarity**.
- Top-k kasus dengan skor kemiripan tertinggi dipilih sebagai hasil retrieval.
- Pendekatan ini bersifat **unsupervised** dan murni berbasis kemiripan teks.

#### b. TF-IDF + Support Vector Machine (SVM) (Pendekatan Supervised Classification)

- Menggunakan **TF-IDF vectorizer** untuk mengubah ringkasan fakta menjadi fitur numerik.
- Menggunakan model **LinearSVC (SVM)** dari `sklearn` yang dilatih secara supervised dengan `case_id` sebagai label target.
- Model mempelajari pola dari data latih dan digunakan untuk memprediksi satu kasus (case_id) yang paling cocok dengan query baru.
- Pendekatan ini bersifat **supervised learning** dan menekankan pada klasifikasi.

Kedua pendekatan digunakan untuk saling melengkapi dalam proses evaluasi performa sistem pada tahap selanjutnya.

---

### 4. Fungsi Retrieval

Dua fungsi `retrieve()` disiapkan, masing-masing untuk kedua pendekatan:

- Pada **pendekatan TF-IDF + Cosine**, fungsi `retrieve(query: str, k: int = 5)` akan:
  1. Mengubah query menjadi vektor TF-IDF.
  2. Menghitung cosine similarity dengan seluruh vektor kasus.
  3. Mengembalikan **top-k case_id** dengan skor tertinggi.

- Pada **pendekatan TF-IDF + SVM**, fungsi `retrieve(query: str, k: int = 1)` akan:
  1. Mengubah query menjadi vektor TF-IDF.
  2. Melakukan klasifikasi menggunakan model SVM.
  3. Mengembalikan **case_id hasil prediksi** dari model (top-1).

---

Dengan pendekatan ganda ini, sistem mampu membandingkan efektivitas metode retrieval berbasis kemiripan teks dengan metode klasifikasi berbasis pembelajaran mesin.


### 5. Pengujian Awal

- Disiapkan **10 query uji** beserta **ground truth** (case ID yang dianggap paling relevan).
- Query dan ground truth disimpan ke file `data/eval/queries.json` untuk keperluan evaluasi pada tahap selanjutnya.

---

## Output

- Model klasifikasi berbasis SVM disimpan di:  
  `03_retrieval_model.pkl`
- Vectorizer TF-IDF disimpan di:  
  `03_vectorizer.pkl`
- Dataset query uji disimpan di:  
  `data/eval/queries.json`

Contoh output terminal:

[SUKSES] Tahap 3 Case Retrieval selesai:

Model disimpan di : 03_retrieval_model.pkl

Vectorizer disimpan di : 03_vectorizer.pkl

10 query uji disimpan di : data/eval/queries.json


---

Tahap Case Retrieval telah berhasil diimplementasikan menggunakan pendekatan supervised classification berbasis **TF-IDF + SVM**. Model ini siap digunakan untuk tahap prediksi (Solution Reuse) dan evaluasi performa pada tahap selanjutnya.


In [12]:
import os
import json
import numpy as np
import pandas as pd
from typing import List
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
import joblib

# === PATH KONFIGURASI ===
DATA_JSON = "data/processed/cases_extracted.json"
MODEL_OUTPUT = "03_retrieval_model.pkl"
VECTORIZER_OUTPUT = "03_vectorizer.pkl"
EVAL_QUERIES = "data/eval/queries.json"
os.makedirs("data/eval", exist_ok=True)

# === LOAD DATASET JSON ===
with open(DATA_JSON, encoding='utf-8') as f:
    data = json.load(f)

case_ids = [case["case_id"] for case in data]
texts = [case["ringkasan_fakta"] for case in data]

# === SPLIT DATA 80:20 ===
X_train, X_test, y_train, y_test = train_test_split(
    texts, case_ids, test_size=0.2, random_state=42
)

# === TRAIN TF-IDF + SVM ===
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
classifier = LinearSVC()
classifier.fit(X_train_vec, y_train)

# === SIMPAN MODEL DAN VECTORIZER ===
joblib.dump(classifier, MODEL_OUTPUT)
joblib.dump(vectorizer, VECTORIZER_OUTPUT)

# === FUNGSI RETRIEVE MENGGUNAKAN MODEL SVM ===
def retrieve(query: str, k: int = 1) -> List[int]:
    query_vec = vectorizer.transform([query])
    prediction = classifier.predict(query_vec)
    return prediction.tolist()

# === PENGUJIAN AWAL: 10 QUERY UJI ===
sample_queries = [
    {
        "query": "seorang pria membunuh ibunya dengan pisau lalu menguburkan jasadnya",
        "ground_truth": [1]
    },
    {
        "query": "penusukan karena perselingkuhan dengan istri terdakwa menyebabkan korban tewas",
        "ground_truth": [3]
    },
    {
        "query": "perkelahian karena perebutan lahan parkir berujung kematian",
        "ground_truth": [2]
    },
    {
        "query": "terdakwa berkelahi dengan temannya di salah satu tempat makan karena perkara sepeda motor",
        "ground_truth": [4]
    },
    {
        "query": "terdakwa membunuh istrinya karena hubungan seksual yang sangat brutal dan juga sedang dalam pengaruh narkoba",
        "ground_truth": [5]
    },
    {
        "query": "terdakwa membunuh rekannya karena perkara pembagian uang pada suatu kafe",
        "ground_truth": [6]
    },
    {
        "query": "terdakwa membunuh korban yaitu tuan rumah ketika sedang melakukan pencurian dirumahnya",
        "ground_truth": [7]
    },
    {
        "query": "terdakwa membunuh karena dendam sering dituduh mencuri di kost mahasiswa",
        "ground_truth": [8]
    },
    {
        "query": "terdakwa membunuh pemilik kost karena diancam diusir akibat penunggakan pembayaran kost",
        "ground_truth": [9]
    },
    {
        "query": "terdakwa membunuh korban karena keduanya sedang dalam kondisi mabuk",
        "ground_truth": [10]
    }
]

# === SIMPAN QUERY UJI KE FILE JSON ===
with open(EVAL_QUERIES, "w", encoding="utf-8") as f:
    json.dump(sample_queries, f, indent=2, ensure_ascii=False)

print("[SUKSES] Tahap 3 Case Retrieval selesai:")
print(f"- Model disimpan di        : {MODEL_OUTPUT}")
print(f"- Vectorizer disimpan di   : {VECTORIZER_OUTPUT}")
print(f"- 10 query uji disimpan di : {EVAL_QUERIES}")

[SUKSES] Tahap 3 Case Retrieval selesai:
- Model disimpan di        : 03_retrieval_model.pkl
- Vectorizer disimpan di   : 03_vectorizer.pkl
- 10 query uji disimpan di : data/eval/queries.json


### Tahap 4 – Solution Reuse

#### Tujuan
Pada tahap ini, sistem bertujuan untuk memanfaatkan solusi dari kasus lama (putusan pengadilan) sebagai referensi atau dasar prediksi terhadap kasus baru yang serupa.

---

#### Langkah Kerja

1. **Ekstraksi Solusi**
   - Dari setiap kasus lama yang diretriev, sistem mengambil **teks amar putusan** atau **isi putusan lengkap** sebagai bentuk solusi.
   - Solusi disimpan dalam struktur dictionary dengan format `{case_id: solusi_text}`.

2. **Algoritma Prediksi**
   Dua metode prediksi solusi diimplementasikan berdasarkan pendekatan yang digunakan:

   - **TF-IDF + Cosine Similarity (CBR)**
     - Sistem melakukan pengambilan top-k kasus paling relevan berdasarkan cosine similarity terhadap vektor TF-IDF.
     - Solusi dari kasus pertama (top-1) dianggap paling representatif dan diambil sebagai `predicted_solution`.

   - **TF-IDF + SVM (Supervised Classification)**
     - Sistem memprediksi satu case ID (top-1) menggunakan model klasifikasi SVM.
     - Amar putusan dari kasus hasil prediksi diambil sebagai `predicted_solution`.

3. **Ringkasan Solusi**
   - Untuk memastikan keterbacaan dan penyajian ringkas, solusi yang diprediksi disingkat seperti abstrak (sekitar 50 kata pertama).

4. **Demo Manual**
   - Terdapat 10 query kasus baru yang digunakan sebagai uji coba.
   - Untuk masing-masing query, sistem menjalankan fungsi `predict_outcome()` dan membandingkan solusi yang diprediksi dengan konteks masalah.

---

#### Fungsi Utama

- `retrieve(query: str, k: int = 5)`: Mengambil top-k `case_id` berdasarkan pendekatan yang digunakan (Cosine atau SVM).
- `predict_outcome(query: str) -> Tuple[str, List[int]]`: Mengembalikan ringkasan solusi prediksi dan daftar case_id yang digunakan sebagai referensi.

---

#### Output

Dua file hasil prediksi disimpan dalam direktori:

- `data/results/predictions_cosine.csv`: Berisi hasil prediksi menggunakan **TF-IDF + Cosine Similarity**.
- `data/results/predictions_svm.csv`: Berisi hasil prediksi menggunakan **TF-IDF + SVM**.

Setiap file prediksi mencakup kolom:
- `query_id`: Nomor urut query.
- `query`: Ringkasan kasus baru.
- `predicted_solution`: Ringkasan solusi yang diprediksi dari kasus lama.
- `top_5_case_ids`: Daftar `case_id` dari kasus lama yang dijadikan referensi.

Contoh struktur isi file prediksi:

query_id,query,predicted_solution,top_5_case_ids
1,seorang pria membunuh ibunya dengan pisau...,putusan nomor 1323/pid.b/2024/pn mdn..., [6, 30, 1, 41, 20]
2,penusukan karena perselingkuhan...,putusan nomor 1100/pid.b/2024/pn mdn..., [3, 15, 5, 42, 47]
...


---

#### Catatan
Dengan adanya dua pendekatan (unsupervised dan supervised), sistem dapat dibandingkan performanya pada tahap evaluasi berikutnya untuk melihat model mana yang paling efektif dalam memanfaatkan solusi dari kasus terdahulu.


In [17]:
import os
import json
import joblib
import pandas as pd
from typing import List, Dict
from sklearn.metrics.pairwise import cosine_similarity

# === PATH ===
DATA_JSON = "data/processed/cases_extracted.json"
MODEL_PATH = "03_retrieval_model.pkl"
VECTORIZER_PATH = "03_vectorizer.pkl"
EVAL_QUERIES = "data/eval/queries.json"
PREDICTION_CSV = "data/results/predictions_cosine.csv"
os.makedirs("data/results", exist_ok=True)

# === LOAD MODEL DAN VECTORIZER ===
model = joblib.load(MODEL_PATH)
vectorizer = joblib.load(VECTORIZER_PATH)

# === LOAD CASE DATA ===
with open(DATA_JSON, encoding='utf-8') as f:
    data = json.load(f)

# Buat struktur case_id → solusi (di sini pakai text_full sebagai solusi)
case_solutions: Dict[int, str] = {case["case_id"]: case["text_full"] for case in data}

# Siapkan data TF-IDF untuk semua kasus
texts = [case["ringkasan_fakta"] for case in data]
case_ids = [case["case_id"] for case in data]
X_all = vectorizer.transform(texts)

# === RETRIEVE DENGAN COSINE SIMILARITY BERDASARKAN TF-IDF MODEL ===
def retrieve(query: str, k: int = 5) -> List[int]:
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, X_all).flatten()
    top_indices = sims.argsort()[::-1][:k]
    return [case_ids[i] for i in top_indices]

# === SOLUTION REUSE FUNCTION ===
def predict_outcome(query: str, k: int = 5) -> str:
    top_k = retrieve(query, k)
    solutions = [case_solutions[cid] for cid in top_k]
    most_common = solutions[0]
    # Ringkas solusi seperti abstrak
    predicted_summary = " ".join(most_common.split()[:50]) + "..."
    return predicted_summary, top_k

# === MUAT QUERY UJI ===
with open(EVAL_QUERIES, encoding='utf-8') as f:
    eval_queries = json.load(f)

# === PROSES DAN SIMPAN HASIL ===
prediction_rows = []
for i, item in enumerate(eval_queries, 1):
    query = item["query"]
    predicted_solution, top_5_ids = predict_outcome(query)
    short_query = query[:100] + "..." if len(query) > 100 else query
    prediction_rows.append({
        "query_id": i,
        "query": short_query,
        "predicted_solution": predicted_solution,
        "top_5_case_ids": top_5_ids
    })

# Simpan ke CSV
pd.DataFrame(prediction_rows).to_csv(PREDICTION_CSV, index=False)

print("[SUKSES] Tahap 4 - Solution Reuse selesai")
print(f"- Hasil prediksi disimpan di: {PREDICTION_CSV}")

[SUKSES] Tahap 4 - Solution Reuse selesai
- Hasil prediksi disimpan di: data/results/predictions_cosine.csv


In [18]:
import os
import json
import joblib
import pandas as pd
from typing import List, Dict

# === PATH ===
DATA_JSON = "data/processed/cases_extracted.json"
MODEL_PATH = "03_retrieval_model.pkl"
VECTORIZER_PATH = "03_vectorizer.pkl"
EVAL_QUERIES = "data/eval/queries.json"
PREDICTION_CSV = "data/results/predictions_svm.csv"
os.makedirs("data/results", exist_ok=True)

# === LOAD MODEL DAN VECTORIZER ===
model = joblib.load(MODEL_PATH)
vectorizer = joblib.load(VECTORIZER_PATH)

# === LOAD CASE DATA ===
with open(DATA_JSON, encoding='utf-8') as f:
    data = json.load(f)

# Buat struktur case_id → solusi (di sini pakai text_full sebagai solusi)
case_solutions: Dict[int, str] = {case["case_id"]: case["text_full"] for case in data}

# Siapkan data untuk indexing
texts = [case["ringkasan_fakta"] for case in data]
case_ids = [case["case_id"] for case in data]

# === RETRIEVE DENGAN SVM ===
def retrieve(query: str) -> List[int]:
    query_vec = vectorizer.transform([query])
    prediction = model.predict(query_vec)  # klasifikasi supervised, hasil satu case_id
    return prediction.tolist()  # hasilnya list dengan 1 item

# === SOLUTION REUSE FUNCTION ===
def predict_outcome(query: str) -> str:
    top_1 = retrieve(query)  # hasil dari model SVM
    solutions = [case_solutions[cid] for cid in top_1]
    predicted_summary = " ".join(solutions[0].split()[:50]) + "..."  # Ringkas abstrak
    return predicted_summary, top_1

# === MUAT QUERY UJI ===
with open(EVAL_QUERIES, encoding='utf-8') as f:
    eval_queries = json.load(f)

# === PROSES DAN SIMPAN HASIL ===
prediction_rows = []
for i, item in enumerate(eval_queries, 1):
    query = item["query"]
    predicted_solution, top_ids = predict_outcome(query)
    short_query = query[:100] + "..." if len(query) > 100 else query
    prediction_rows.append({
        "query_id": i,
        "query": short_query,
        "predicted_solution": predicted_solution,
        "top_5_case_ids": top_ids
    })

# Simpan ke CSV
pd.DataFrame(prediction_rows).to_csv(PREDICTION_CSV, index=False)

print("[SUKSES] Tahap 4 - Solution Reuse selesai")
print(f"- Hasil prediksi disimpan di: {PREDICTION_CSV}")


[SUKSES] Tahap 4 - Solution Reuse selesai
- Hasil prediksi disimpan di: data/results/predictions_svm.csv


## Tahap 5 – Model Evaluation

### Tujuan
Tahap ini bertujuan untuk mengukur dan menganalisis performa sistem dalam melakukan _retrieval_ dan prediksi solusi atas kasus baru berdasarkan kasus-kasus lama.

---

### Langkah Kerja

#### 1. Evaluasi Retrieval
Evaluasi dilakukan terhadap hasil retrieval menggunakan dua pendekatan:
- **TF-IDF + Cosine Similarity (unsupervised/CBR)**
- **TF-IDF + SVM (supervised classification)**

Metrik yang digunakan:
- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**

Evaluasi dilakukan dengan membandingkan `top_k` hasil prediksi terhadap ground-truth `case_id` dari 10 query uji.

#### 2. Visualisasi & Laporan
- Tabel perbandingan metrik antar model ditampilkan sebagai grafik batang (_bar chart_).
- Analisis terhadap **kasus-kasus gagal (error analysis)** juga dilakukan dan disimpan dalam format `.json`.

---

### Implementasi

Fungsi evaluasi utama meliputi:
- `eval_retrieval()`: Mengevaluasi pendekatan berbasis _cosine similarity_ (top-k match).
- `eval_prediction()`: Mengevaluasi prediksi top-1 dari model SVM.
- `save_errors()`: Menyimpan daftar query yang gagal diprediksi dengan benar.

---

### Output

Berikut merupakan ringkasan hasil evaluasi model:

#### 1. File Hasil Evaluasi

- **Retrieval (Cosine Similarity)**  
  Disimpan pada: `data/eval/retrieval_metrics.csv`
  
model,accuracy,precision,recall,f1_score
TF-IDF + Cosine,0.9,1.0,0.9,0.9473684210526315


- **Prediksi (SVM)**  
Disimpan pada: `data/eval/prediction_metrics.csv`

model,accuracy,precision,recall,f1_score
TF-IDF + SVM,0.6,1.0,0.6,0.75

#### 2. Visualisasi

Grafik perbandingan performa antar model disimpan di:
data/eval/performance_comparison.png


#### 3. Error Analysis

- **Kesalahan pada TF-IDF + Cosine:**
  Disimpan di: `data/eval/error_cases_cosine.json`
  ```json
  [
    {
      "query_id": 7,
      "query": "terdakwa membunuh korban yaitu tuan rumah ketika sedang melakukan pencurian dirumahnya",
      "predicted": [5, 20, 1, 43, 18],
      "ground_truth": [7]
    }
  ]


- **Kesalahan pada TF-IDF + SVM:**
Disimpan di: `data/eval/error_cases_svm.json`
  ```json
[
  {
    "query_id": 1,
    "query": "seorang pria membunuh ibunya dengan pisau lalu menguburkan jasadnya",
    "predicted": [6],
    "ground_truth": [1]
  },
  {
    "query_id": 4,
    "query": "terdakwa berkelahi dengan temannya di salah satu tempat makan karena perkara sepeda motor",
    "predicted": [45],
    "ground_truth": [4]
  },
  {
    "query_id": 5,
    "query": "terdakwa membunuh istrinya karena hubungan seksual yang sangat brutal dan juga sedang dalam pengaruh...",
    "predicted": [3],
    "ground_truth": [5]
  },
  {
    "query_id": 7,
    "query": "terdakwa membunuh korban yaitu tuan rumah ketika sedang melakukan pencurian dirumahnya",
    "predicted": [16],
    "ground_truth": [7]
  }
]


In [26]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from typing import List, Dict

# === PATH ===
COSINE_PRED = "data/results/predictions_cosine.csv"
SVM_PRED = "data/results/predictions_svm.csv"
EVAL_QUERIES = "data/eval/queries.json"
RETRIEVAL_METRIC_OUTPUT = "data/eval/retrieval_metrics.csv"
PREDICTION_METRIC_OUTPUT = "data/eval/prediction_metrics.csv"
ERROR_COSINE_OUTPUT = "data/eval/error_cases_cosine.json"
ERROR_SVM_OUTPUT = "data/eval/error_cases_svm.json"
os.makedirs("data/eval", exist_ok=True)

# === LOAD GROUND TRUTH ===
with open(EVAL_QUERIES, encoding='utf-8') as f:
    ground_truth_data = json.load(f)
    gt_dict = {i+1: item["ground_truth"] for i, item in enumerate(ground_truth_data)}

# === EVALUASI RETRIEVAL (COSINE: TOP-K MATCHING) ===
def eval_retrieval(pred_file: str, model_name: str, k: int = 5) -> Dict:
    df = pd.read_csv(pred_file)
    y_true = []
    y_pred = []
    for _, row in df.iterrows():
        query_id = int(row["query_id"])
        pred_ids = eval(row["top_5_case_ids"])[:k]
        gt = gt_dict[query_id]
        hit = any(pid in gt for pid in pred_ids)
        y_true.append(1)
        y_pred.append(1 if hit else 0)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return {
        "model": model_name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1
    }

# === EVALUASI PREDIKSI SVM SEBAGAI RETRIEVAL (TOP-1 MATCH) ===
def eval_prediction(pred_file: str, model_name: str) -> Dict:
    df = pd.read_csv(pred_file)
    y_true = []
    y_pred = []
    for _, row in df.iterrows():
        query_id = int(row["query_id"])
        pred_ids = eval(row["top_5_case_ids"])
        pred = pred_ids[0] if pred_ids else -1
        gt = gt_dict[query_id]
        hit = pred in gt
        y_true.append(1)
        y_pred.append(1 if hit else 0)

    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred)
    }

# === SIMPAN KASUS GAGAL (ERROR ANALYSIS) ===
def save_errors(pred_file: str, output_file: str):
    df = pd.read_csv(pred_file)
    error_cases = []
    for _, row in df.iterrows():
        query_id = int(row["query_id"])
        pred_ids = eval(row["top_5_case_ids"])
        gt = gt_dict[query_id]
        if not any(pid in gt for pid in pred_ids):
            error_cases.append({
                "query_id": query_id,
                "query": row["query"],
                "predicted": pred_ids,
                "ground_truth": gt
            })
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(error_cases, f, indent=2, ensure_ascii=False)

# === JALANKAN EVALUASI ===
retrieval_metrics = eval_retrieval(COSINE_PRED, "TF-IDF + Cosine")
prediction_metrics = eval_prediction(SVM_PRED, "TF-IDF + SVM")

# === SIMPAN METRIK KE FILE TERPISAH ===
pd.DataFrame([retrieval_metrics]).to_csv(RETRIEVAL_METRIC_OUTPUT, index=False)
pd.DataFrame([prediction_metrics]).to_csv(PREDICTION_METRIC_OUTPUT, index=False)

# === SIMPAN KASUS GAGAL ===
save_errors(COSINE_PRED, ERROR_COSINE_OUTPUT)
save_errors(SVM_PRED, ERROR_SVM_OUTPUT)

# === VISUALISASI ===
combined_df = pd.DataFrame([retrieval_metrics, prediction_metrics])
combined_df.set_index("model")[["accuracy", "precision", "recall", "f1_score"]].plot(
    kind="bar", figsize=(10, 6), title="Perbandingan Model"
)
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("data/eval/performance_comparison.png")
plt.close()

print("[SUKSES] Tahap 5 - Evaluasi Model selesai")
print(f"- Hasil evaluasi retrieval disimpan di: {RETRIEVAL_METRIC_OUTPUT}")
print(f"- Hasil evaluasi prediksi disimpan di : {PREDICTION_METRIC_OUTPUT}")
print("- Visualisasi disimpan di: data/eval/performance_comparison.png")
print(f"- Error cosine disimpan di: {ERROR_COSINE_OUTPUT}")
print(f"- Error svm disimpan di: {ERROR_SVM_OUTPUT}")


[SUKSES] Tahap 5 - Evaluasi Model selesai
- Hasil evaluasi retrieval disimpan di: data/eval/retrieval_metrics.csv
- Hasil evaluasi prediksi disimpan di : data/eval/prediction_metrics.csv
- Visualisasi disimpan di: data/eval/performance_comparison.png
- Error cosine disimpan di: data/eval/error_cases_cosine.json
- Error svm disimpan di: data/eval/error_cases_svm.json


## Kesimpulan

Proyek ini telah berhasil membangun sebuah sistem **Case-Based Reasoning (CBR)** untuk menganalisis putusan pengadilan menggunakan pendekatan berbasis **representasi teks TF-IDF**. Proyek ini melibatkan lima tahap utama, yaitu:

1. **Pembangunan Case Base**: Berhasil dilakukan scraping dan pembersihan lebih dari 40 dokumen putusan pidana dari Direktori Putusan MA, disimpan dalam format `.txt` setelah melalui validasi integritas isi.
2. **Case Representation**: Setiap kasus direpresentasikan dalam struktur terorganisir, mencakup metadata, ringkasan fakta, pasal, pihak, dan konten penuh. Proses feature engineering dilakukan melalui analisis panjang teks, Bag-of-Words, dan QA-pairs.
3. **Case Retrieval**: Dua pendekatan retrieval dibangun dan dibandingkan:
   - **TF-IDF + Cosine Similarity** 
   - **TF-IDF + SVM (LinearSVC)** 
4. **Solution Reuse**: Sistem mampu memprediksi solusi dari kasus baru dengan mengambil solusi dari kasus serupa teratas. Fungsi `predict_outcome` berhasil mengeluarkan ringkasan putusan sebagai prediksi solusi.
5. **Model Evaluation**: Evaluasi kuantitatif dilakukan terhadap kedua pendekatan menggunakan metrik akurasi, presisi, recall, dan F1-score. Hasil menunjukkan bahwa pendekatan TF-IDF + Cosine memiliki performa lebih stabil dengan F1-score tertinggi, sedangkan pendekatan TF-IDF + SVM lebih rentan terhadap generalisasi kasus baru.

| Model              | Accuracy | Precision | Recall | F1-score |
|--------------------|----------|-----------|--------|----------|
| TF-IDF + Cosine    | 0.90     | 1.00      | 0.90   | 0.95     |
| TF-IDF + SVM       | 0.60     | 1.00      | 0.60   | 0.75     |

Visualisasi performa dan analisis kasus kegagalan memperkuat bukti bahwa pendekatan berbasis similarity lebih sesuai untuk domain CBR ini, terutama ketika data pelatihan terbatas atau tidak seimbang.

---

## Penutup

Melalui proyek ini, kami memahami secara menyeluruh bagaimana proses reasoning berbasis kasus dapat dibangun menggunakan pendekatan NLP dan machine learning. Integrasi antara representasi tekstual dan algoritma retrieval memberikan solusi yang praktis untuk mendukung analisis yurisprudensi secara komputasional. Ke depan, sistem ini dapat dikembangkan lebih lanjut dengan pendekatan embedding berbasis BERT atau dengan integrasi ke basis data yudisial berskala nasional.

---
